In [24]:
import json
import pandas as pd
import re

from bs4 import BeautifulSoup

from langdetect import detect
from langdetect import DetectorFactory

DetectorFactory.seed = 42

with open("../data/raw_ads_2511.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

## Datakvalitetskontroll inför städning

In [25]:
print("Antal annonser:", len(df))
print("Unika annons-ID:", df["id"].nunique())

Antal annonser: 6889
Unika annons-ID: 6632


In [26]:
def detect_language(text):
    if not text or not text.strip():
        return "unknown"

    try:
        return detect(text)
    except:
        return "unknown"


# Detektera språk för varje annons
df["language"] = df["description"].apply(detect_language)


# Visa fördelningen av språk
print("Språkfördelning:")
print(df["language"].value_counts())


Språkfördelning:
language
sv         4424
en         2455
unknown       8
fr            2
Name: count, dtype: int64


In [27]:
# Undersöker annonser med språk "unknown"
unknown_ads = df[df["language"] == "unknown"]

for _, row in unknown_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Text:", row["description"])

ID: 29506534
Titel: Teamcenter Solution Architect operations
Text: 
ID: 29737028
Titel: Core Network Analyst
Text: 
ID: 29817808
Titel: Senior Power BI Specialist - Älmhult
Text: 
ID: 30456047
Titel: UI/UX designer
Text: 
ID: 30600803
Titel: Dynamics 365 Architect / D365 Solution Architect  16004
Text: 
ID: 30854632
Titel: Power BI Consultant (Report Development)
Text: 
ID: 30961171
Titel: Senior Data Engineer
Text: 
ID: 30983013
Titel: Linux Engineer
Text: 


In [28]:
# Identifierar annonser som saknar description eller endast innehåller tomrum
missing_description = df[
    df["description"].isna() |
    df["description"].fillna("").str.strip().eq("")
]

print("Antal annonser utan description:", len(missing_description))

missing_description[
    ["id", "title", "employer", "occupation", "description"]
].head(10)

Antal annonser utan description: 8


,id,title,employer,occupation,description
931,29506534,Teamcenter Solution Architect operations,Hitachi Energy Sweden AB,IT-arkitekt/Lösningsarkitekt,
2424,29737028,Core Network Analyst,Semicon Service Nordic AB,Systemanalytiker/Systemutredare,
2824,29817808,Senior Power BI Specialist - Älmhult,Danda AB,"Affärskonsult, IT",
4475,30456047,UI/UX designer,GreenBay GreenTech Nordic AB,IT-utredare,
4903,30600803,Dynamics 365 Architect / D365 Solution Archite...,Veritaz AB,Enterprise Architect,
5455,30854632,Power BI Consultant (Report Development),Semicon Service Nordic AB,"Verksamhetskonsult, IT",
5776,30961171,Senior Data Engineer,Semicon Service Nordic AB,Data scientist,
5824,30983013,Linux Engineer,Ericsson AB,Systemanalytiker/Systemutredare,


In [29]:
# Identifierar alla rader som tillhör ett duplicerat annons-ID
duplicate_ids = df[
    df["id"].duplicated(keep=False)
].sort_values("id")

print("Antal rader som tillhör ett duplicerat ID:", len(duplicate_ids))
print("Antal duplicerade ID:", duplicate_ids["id"].nunique())

duplicate_ids[
    ["id", "version_id", "title", "employer", "published"]
].head(10)


Antal rader som tillhör ett duplicerat ID: 475
Antal duplicerade ID: 218


,id,version_id,title,employer,published
4,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,2025-01-02T10:15:14
5,29354585,f0883b87209ad585259381a396b05e5618ea3d93,Projektledare,Randstad AB,2025-01-02T10:15:14
30,29361752,27b784e84eb1a5703befb304f9f76c3a3c8797db,Teknikansvarig VA till SYSTRA AB i Jönköping,SYSTRA AB,2025-01-07T13:27:42
31,29361752,5fddf1db951f3374f7792910c4f425650ce236e6,Teknikansvarig VA till SYSTRA i Jönköping,SYSTRA AB,2025-01-07T13:27:42
37,29362368,98e4dc9c1ec0419c0fcef0319b64909342df680d,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
38,29362368,aa58e8f5ebbe56dd9dbc558eea70eb14085eb6de,Lösningsarkitekt Nätverksinfrastruktur,Försvarsmakten,2025-01-07T15:11:03
39,29362443,4acb1a3da382edaf553e92b57524ad381d704d85,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
40,29362443,a335ce8a599380486cb101644acedcc27bef5d5d,Lösningsarkitekt IT-säkerhet,Försvarsmakten,2025-01-07T15:28:34
50,29363892,de9e985fb72fb75014b4c4f676ce7c8bee93db9e,Senior Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04
51,29363892,e3e83d3e1186b260000ba39f70d40dbb5265842d,Frontend Engineer,Schibsted Sverige AB,2025-01-08T09:56:04


In [30]:
# Identifierar HTML-taggar som kan utgöra brus i NLP-analysen
html_pattern = r"<[^>]+>"

html_ads = df[
    df["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal annonser med möjlig HTML:", len(html_ads))

html_ads[
    ["id", "title", "description"]
].head(10)

Antal annonser med möjlig HTML: 3


,id,title,description
4717,30542907,Infrastrukturarkitekt till Växjö kommun,Vill du arbeta med teknik som gör verklig skil...
4966,30622455,Chefsarkitekt | Jula | Skara,Är du redo att forma Julas digitala framtid? D...
5606,30911356,Lösningsarkitekt,Vill du vara med och forma framtidens digitala...


In [31]:
# Regex för att identifiera personnummer
personnummer_pattern = r"\b(?:19|20)?\d{6}[-+ ]?\d{4}\b"

personnummer_count = df["description"].fillna("").str.contains(
    personnummer_pattern, regex=True
).sum()

print("Annonser med möjliga personnummer:", personnummer_count)

Annonser med möjliga personnummer: 0


In [32]:
# Regex för att identifiera epost-adresser
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

email_ads = df[
    df["description"]
    .fillna("")
    .str.contains(email_pattern, regex=True)
]

print("Antal annonser med e-post:", len(email_ads))


Antal annonser med e-post: 0


In [33]:
# Regex för att identifiera möjliga svenska telefonnummer
phone_pattern = r"(?<!\d)(?:\+46|0046|0)[\s-]?(?:\d[\s-]?){7,11}(?!\d)"

phone_ads = df[
    df["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print("Antal annonser med möjliga telefonnummer:", len(phone_ads))


Antal annonser med möjliga telefonnummer: 24


In [34]:
# Kontroll av möjliga telefonnummer före rensning
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

phone_results = df.loc[
    phone_matches.str.len() > 0,
    ["id", "title", "employer"]
].copy()

phone_results["phone_match"] = phone_matches[
    phone_matches.str.len() > 0
].apply(lambda x: ", ".join(x))

phone_results

,id,title,employer,phone_match
186,29386902,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
187,29386904,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
188,29386905,IT-arkitekt till domänen Business Common,Vattenfall AB,02-02-2025
427,29422415,Senior Kravanalytiker,Deploja AB,03-02-2025\n
535,29439711,"IT-arkitekt, inriktning Digital Arbetsplats",Länsstyrelsen i Västra Götalands län,000-59 000
1045,29533240,Dataanalytiker sökes av Göteborgs Spårvägar,Göteborgs Stads Kollektivtrafik AB,000-52 000
1865,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
1866,29663343,IT Specialist till VB Energi,AFRY AB,0240-876 00
2667,29791470,Senior IT-arkitekt,Deploja AB,01-07-2025
3048,29929162,IT Systemförvaltare – Teknikstöd nät och platt...,Försvarsmakten,0589-40400


In [35]:
# Identifierar möjliga telefonnummer i annonstexterna med hjälp av regex
phone_matches = df["description"].fillna("").str.findall(phone_pattern)

# Hämtar de annonser där minst ett möjligt telefonnummer har hittats
phone_ads = df.loc[
    phone_matches.str.len() > 0
].copy()

# Visar telefonträffarna tillsammans med texten runt träffen för manuell granskning
for _, row in phone_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print("Träff:", phone_matches.loc[row.name])
    
    description = row["description"]
    
    for match in phone_matches.loc[row.name]:
        start = description.find(match)
        context = description[max(0, start - 100): start + len(match) + 100]
        print("Sammanhang:")
        print(context)

ID: 29386902
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386904
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29386905
Titel: IT-arkitekt till domänen Business Common
Träff: ['02-02-2025']
Sammanhang:
 föräldraledighetstillägg med mera. Läs mer om våra förmåner här
. Välkommen med din ansökan senast 02-02-2025. Urval och intervjuer sker löpande. Vi tar endast emot ansökningar via vår webbplats. Vi är övertyg
ID: 29422415
Titel: Senior Kravanalytiker
Träff: ['03-02-2025\n']
Sammanhang:
n av

## Cleaning

In [36]:
print("Antal annonser före rensning:", len(df))


# Behåll endast svenska annonser
df_clean = df[df["language"] == "sv"].copy()

print("Antal svenska annonser:", len(df_clean))


Antal annonser före rensning: 6889
Antal svenska annonser: 4424


In [37]:
# Ta bort dubbletter baserat på annons-ID
antal_fore_dubblettrensning = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset="id",
    keep="first"
).copy()

antal_efter_dubblettrensning = len(df_clean)

print("Dubblettrensning:")
print("Antal annonser före dubblettrensning:", antal_fore_dubblettrensning)
print("Antal annonser efter dubblettrensning:", antal_efter_dubblettrensning)
print("Antal borttagna dubbletter:", 
      antal_fore_dubblettrensning - antal_efter_dubblettrensning)

Dubblettrensning:
Antal annonser före dubblettrensning: 4424
Antal annonser efter dubblettrensning: 4264
Antal borttagna dubbletter: 160


In [38]:
# Identifierar annonser som saknar annonstext
missing_description = df_clean[
    df_clean["description"].isna() |
    df_clean["description"].fillna("").str.strip().eq("")
]

print("Annonser utan description efter rensning:", len(missing_description))

Annonser utan description efter rensning: 0


In [39]:
# Kontrollerar om möjliga telefonnummer fortfarande finns kvar efter städning
phone_ads_after_duplicates = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(phone_pattern, regex=True)
]

print(
    "Annonser med möjlig telefonträff efter rensning:",
    len(phone_ads_after_duplicates)
)

Annonser med möjlig telefonträff efter rensning: 23


In [40]:
# Ersätter verifierade telefonnummer med neutral markör 
def clean_phone_numbers(text):
    if pd.isna(text):
        return text
    
    return re.sub(phone_pattern, "[PHONE]", text)


df_clean["description"] = df_clean["description"].apply(clean_phone_numbers)


phone_matches_clean = df_clean["description"].fillna("").str.findall(phone_pattern)

print(
    "Antal möjliga telefonträffar efter rensning:",
    phone_matches_clean.str.len().sum()
)

Antal möjliga telefonträffar efter rensning: 0


In [41]:
# Räknar hur många telefonnummer som ersattes med [PHONE]
phone_replacements = df_clean["description"].fillna("").str.count(r"\[PHONE\]").sum()

print("Antal ersatta telefonnummer:", phone_replacements)

Antal ersatta telefonnummer: 23


In [42]:
# Kontrollerar om några HTML-taggar fortfarande finns kvar efter cleaning
html_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(html_pattern, regex=True)
]

print("Antal HTML-annonser efter rensning:", len(html_ads))

Antal HTML-annonser efter rensning: 3


In [43]:
# Visar annonser med kvarvarande HTML för manuell kontroll
for _, row in html_ads.iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])
    print(row["description"])

ID: 30542907
Titel: Infrastrukturarkitekt till Växjö kommun
Vill du arbeta med teknik som gör verklig skillnad? Hos oss får du chansen att påverka utvecklingen av en hel kommun, där smarta digitala lösningar bidrar till en enklare, tryggare och mer hållbar vardag för invånare, företag och besökare. Nu söker vi en infrastrukturarkitekt som vill vara en nyckelspelare i vår tekniska utvecklingsresa och som motiveras av att arbeta i ett komplext och samhällsviktigt IT‑landskap. Växjö kommun är Kronobergs största arbetsgivare och varje dag arbetar vi tillsammans för framtidens samhälle. <img src="https://sae3ndp007.blob.core.windows.net/swe/71d22a21-1a8c-479b-8fac-9af5f5be811b.jpg">
 
 
Om rollen 
Som infrastrukturarkitekt blir du en del av vårt team Arkitektur & Utveckling, tillsammans med 12 kollegor inom infrastruktur-, nät- och lösningsarkitektur samt systemutveckling. Här får du en central roll i att forma vår tekniska riktning och skapa robusta och säkra lösningar. Du får arbeta i en 

In [44]:
# Tar bort HTML-taggar men behåller den synliga texten
def clean_html(text):
    if pd.isna(text):
        return text
    
    return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)


df_clean["description"] = df_clean["description"].apply(clean_html)


html_after = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
]

print("Annonser med HTML efter cleaning:", len(html_after))

Annonser med HTML efter cleaning: 0


In [45]:
# Söker efter kontaktrelaterade fraser för att manuellt kontrollera
# om personnamn eller andra personuppgifter förekommer i anslutning till dessa
contact_pattern = (
    r"(?i)(kontaktperson|kontakt|kontakta|ansvarig|rekryterare|"
    r"rekryteringsansvarig|för frågor|vid frågor|har du frågor)"
)

contact_ads = df_clean[
    df_clean["description"]
    .fillna("")
    .str.contains(contact_pattern, regex=True)
]

print("Annonser med kontaktfraser:", len(contact_ads))

for _, row in contact_ads.head(20).iterrows():
    description = row["description"]
    match = re.search(contact_pattern, description)

    print("=" * 80)
    print("ID:", row["id"])
    print("Titel:", row["title"])

    if match:
        start = match.start()
        end = min(len(description), match.end() + 200)

        print(description[start:end].replace("\n", " "))

C:\Users\magda\AppData\Local\Temp\ipykernel_44740\2175045008.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(contact_pattern, regex=True)


Annonser med kontaktfraser: 1741
ID: 29354585
Titel: Projektledare
kontaktytor och intressanta utmaningar. Då kan denna tjänst vara något för dig. Beskrivning av gruppen, En hel kommun är beroende av att vi genomför vårt uppdrag i allt från naturnära områden till stadsmiljö
ID: 29355065
Titel: Consultant Manager till ett av våra grymma Dynamics 365 F&SCM-team
kontakt och involveras tidigt i affärerna ur ett resurssättningsperspektiv. Viktigt att poängtera är att vi inte jobbar med resurskonsulter, utan att vi är ett projektkonsultföretag vilket gör att dina medar
ID: 29355172
Titel: Systemspecialist med inriktning IT-infrastruktur till Säkra IT-lösningar
kontakter. Önskar du att arbeta delvis på distans? I den här tjänsten kan det finnas möjlighet att arbeta hemifrån upp till 40 procent av din arbetstid. #LI-Hybrid Intervjuer sker via Skype, preliminärt unde
ID: 29355229
Titel: IT arkitekt / Lösningsarkitekt till energisektorn i Umeå
ansvarig för Lösningsarkitekturteamet. Uppdragsperio

- För att identifiera annonsernas språk användes paketet `langdetect`. Endast annonser som klassificerades som svenska behölls för den fortsatta analysen. De annonser som klassificerades som `unknown` granskades manuellt och visade sig bestå av tomma annonser. Dessa ingår därför inte i den fortsatta kompetensanalysen.

- Vid kontroll av potentiella personuppgifter identifierades inga träffar för e-postadresser eller personnummer.  

- Regexbaserad identifiering av telefonnummer gav 23 möjliga träffar. Dessa granskades manuellt för att bedöma om de utgjorde faktiska telefonnummer.   
Några av träffarna visade sig vara telefonnummer till företagsväxlar. Verifierade telefonnummer, inklusive företagsväxlar, ersätts med `[PHONE]` eftersom de inte är relevanta för kompetensanalysen.

- Förekomst av HTML-taggar kontrollerades med regex. Ett fåtal annonser innehöll HTML-formatering, men mängden var begränsad. HTML-taggarna tas bort eftersom de inte innehåller relevant information för kompetensanalysen och annars kan utgöra brus vid den fortsatta NLP-bearbetningen.

- Som en extra kontroll av personnamn söktes kontaktrelaterade fraser i annonstexterna för att undersöka om man kan hitta personnamn. Vid manuell granskning syntes inga direkta personuppgifter, utan i de fall där sådan information verkar ha funnits var den redan ersatt med `****`.     
Det tyder på att personuppgifter redan har maskerats i materialet som hämtats från JobTech Historical API.

- Regexbaserad identifiering kan användas för strukturerade uppgifter som e-postadresser, telefonnummer och personnummer men är mindre lämplig för att identifiera personnamn. Förekomst av personnamn kommer därför undersökas separat med spaCy NER. 

